# Part 5 — Anomaly Detection & Explainability

**Industrial Predictive Maintenance & Sensor Degradation Analysis**

This notebook covers the unsupervised anomaly detection pipeline, model explainability with SHAP, cost analysis, and sensor drift simulation.
The core implementation script is located at `src/models/anomaly_explainability.py`.


## 1. Unsupervised Approach & Data Preparation

- In real industrial settings, machine failures are rare and novel failure types may emerge.
- An **Autoencoder** is trained strictly on **normal (non-failure) operating records**.
- When given new sensor data, the Autoencoder reconstructs the inputs. Abnormal conditions or degradation yield higher reconstruction errors (MSE).
- No leakage: same stratified split as Parts 3 and 4 (random seed 42, 80/20 train/test split).


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

metadata = json.loads((PROJECT_ROOT / "models" / "anomaly_metadata.json").read_text())
metrics_df = pd.read_csv(PROJECT_ROOT / "reports" / "anomaly_metrics.csv")
drift_df = pd.read_csv(PROJECT_ROOT / "reports" / "anomaly_drift_results.csv")

metadata


## 2. Autoencoder Architecture & Training Curves

The autoencoder compresses the 11-dimensional sensor space into an 8-dimensional bottleneck:
- Architecture: `Input(11) -> Dense(32, relu) -> Dense(16, relu) -> Bottleneck(8, relu) -> Dense(16, relu) -> Dense(32, relu) -> Output(11, linear)`
- Optimization: Adam (learning rate 1e-3), MSE loss, early stopping on validation loss.


In [ ]:
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "anomaly_ae_training.png")))


## 3. Reconstruction Error Distribution & Thresholding

The threshold is selected on the training set by minimizing operational cost:
- Cost formula: `Cost = 10 * False Negatives + 1 * False Positives`
- Normal samples concentrate near low error values, whereas actual failures exhibit significantly higher reconstruction errors.


In [ ]:
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "anomaly_error_dist.png")))


## 4. Test Evaluation & Confusion Matrix

Evaluated on the held-out test set (2,000 samples):


In [ ]:
display(metrics_df)
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "anomaly_roc.png")))
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "anomaly_confusion_matrix.png")))


## 5. Model Explainability with SHAP

To explain *why* an instance triggered an anomaly alert, we use SHAP (KernelExplainer) applied to the reconstruction error function.
- Features causing large reconstruction errors are identified as the primary drivers of the anomaly.


In [ ]:
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "anomaly_shap_bar.png")))
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "anomaly_shap_summary.png")))


## 6. Sensor Drift Simulation

We simulated gradual sensor degradation across 10 progressive operating windows:
- Air temperature drifts by +5 K.
- Torque drifts by +15 Nm (simulating wear/friction).
- Derived features are dynamically re-calculated.


In [ ]:
display(drift_df)
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "anomaly_drift_sim.png")))


## 7. Maintenance Strategy Cost Comparison

Comparing overall business cost across strategies:
1. **No Model (Purely Reactive):** Incurs cost for every failure that occurs unannounced.
2. **Autoencoder (Unsupervised Anomaly Detection):** Significantly reduces total cost without requiring failure labels.
3. **Supervised Classifier (Random Forest from Part 3):** Highest performance when historical labeled failures are available.


In [ ]:
display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "anomaly_cost_comparison.png")))


## 8. How to Reproduce Part 5

```bash
python src/models/anomaly_explainability.py
```

This script trains the autoencoder, computes the cost-optimal decision threshold, generates all evaluation figures, runs SHAP explanations, and runs the drift simulation.
